# Imports & Setup

In [1]:
# 1. Install necessary library
!pip install -qU timm

# 2. Imports
import os
import gc
import math
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 44.2 MB/s eta 0:00:00a 0:00:01


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

# Configuration & Seeding

In [2]:
class Config:
    seed = 42
    # EVA-02 Large: A powerful transformer for fine-grained recognition
    model_name = "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k"
    
    img_size = 448
    
    num_classes = 31  # Adjust based on training set unique IDs if needed
    
    # Training Hyperparameters
    num_epochs = 13          # Increased slightly for better convergence
    batch_size = 4           # Keep small for large resolution
    grad_accum = 4           # Effective batch size = 16
    
    # Learning Rates (LLRD specific)
    encoder_lr = 3e-5        # Slower for the backbone
    head_lr = 1e-3           # Faster for the classifier head
    weight_decay = 1e-3      # Standard for ViT
    
    # ArcFace Hyperparameters
    arcface_s = 30.0
    arcface_m = 0.50
    
    # Advanced Options
    train_full_data = False  # Set TRUE for final submission, FALSE for validation
    n_folds = 5
    target_fold = 0          # Which fold to train on if validation is active
    
    use_tta = True           # Test Time Augmentation
    use_dba = True           # Test time DataBase augmentation
    use_qe = True            # Query Expansion
    use_rerank = True        # K-Reciprocal Re-ranking

    base_beta = 0.5
    temperature = 4.0        # Increased for softer distillation
    use_beta_scheduler = False # Toggle for linear ramp-up
    beta_warmup_epochs = 5   # Reaches base_beta at epoch 5
    different_heads = False    # Mix of ArcFace and Linear heads

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(Config.seed)

# Dataset & Advanced Transforms

In [3]:
# Stronger augmentations for the training set
train_transform = transforms.Compose([
    transforms.Resize((Config.img_size, Config.img_size)),
    transforms.RandomResizedCrop(size = (Config.img_size, Config.img_size), scale=(0.5, 0.8)),
    transforms.RandomHorizontalFlip(p=0.5),
    # TrivialAugmentWide: State-of-the-art auto-augmentation for small datasets
    transforms.TrivialAugmentWide(interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25),
])


# Clean transform for validation/testing
test_transform = transforms.Compose([
    transforms.Resize((Config.img_size, Config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
class JaguarDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["filename"]
        img_path = self.img_dir / img_name
        
        try:
            # Using RGB is important for EVA-02 which expects 3 channels
            img = Image.open(img_path).convert("RGB")
        except Exception:
            # Handle missing or corrupted files by returning a blank image
            img = Image.new("RGB", (Config.img_size, Config.img_size))

        if self.transform:
            img = self.transform(img)
            
        if self.is_test:
            return img, img_name
        
        # Return the pre-calculated integer index from our external mapping
        return img, torch.tensor(row["label_idx"], dtype=torch.long)

# Model (EVA-02 + ArcFace + Trainable GeM)

In [5]:
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        # p is now a learnable parameter
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))
        ).pow(1.0 / self.p)

class BalancedArcFace(nn.Module):
    def __init__(self, in_features, out_features, sample_counts, s_base=30.0, m=0.5):
        super().__init__()
        self.s_base = s_base
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        
        # Calculate class-specific scaling factors
        # Logic: s_i = s_base * (total_samples / (n_classes * count_i))
        # We use a log or power transform to prevent extreme scaling
        counts = torch.tensor(sample_counts, dtype=torch.float32)
        avg_count = counts.mean()
        
        # Adaptive factor: rare classes get > 1.0, common get < 1.0
        # We use a square root to dampen the effect so it doesn't explode
        self.register_buffer('s_factors', torch.sqrt(avg_count / counts))
        
    def forward(self, input, label=None):
        # 1. Standard Cosine Similarity
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        
        if label is None:
            return cosine
        
        # 2. Apply ArcFace Margin
        # Clamp for numerical stability (prevents NaNs)
        theta = torch.acos(cosine.clamp(-1.0 + 1e-7, 1.0 - 1e-7))
        phi = torch.cos(theta + self.m)
        
        # 3. Create One-Hot Mask
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1), 1)
        
        # 4. Target class gets phi, others get cosine
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        
        # 5. Apply Class-Dependent Scaling
        # We fetch the specific s_factor for each image in the batch
        batch_s = self.s_base * self.s_factors[label] # Shape: (batch_size,)
        
        # Multiply each row by its specific scale factor
        output = logits * batch_s.view(-1, 1)
        
        return output

class EVABoss(nn.Module):
    def __init__(self, num_classes, class_counts):
        super().__init__()
        self.backbone = timm.create_model(Config.model_name, pretrained=True, num_classes=0)
        self.feat_dim = self.backbone.num_features
        self.gem = GeM()
        self.bn = nn.BatchNorm1d(self.feat_dim)

        # USE BALANCED ARCFACE HERE
        self.head = BalancedArcFace(
            in_features= self.feat_dim, 
            out_features=num_classes, 
            sample_counts=class_counts, # Pass the list of counts per class
            s_base=Config.arcface_s, 
            m=Config.arcface_m
        )

    def forward(self, x, label=None):
        features = self.backbone.forward_features(x)
        
        # Standard reshaping logic for ViT/EVA
        if features.dim() == 3:
            B, N, C = features.shape
            H = W = int(math.sqrt(N))
            if H * W != N: features = features[:, -H*W:, :]
            features = features.permute(0, 2, 1).reshape(B, C, H, W)

        emb = self.gem(features).flatten(1)
        emb = self.bn(emb)
        
        # During training, 'label' is passed to BalancedArcFace to apply the margin
        # During inference (label=None), it returns simple cosine similarities
        return self.head(emb, label)


class CollabEVABoss(nn.Module):
    def __init__(self, num_classes, class_counts, num_heads=1):
        super().__init__()
        self.backbone = timm.create_model(Config.model_name, pretrained=True, num_classes=0)
        self.feat_dim = self.backbone.num_features
        self.gem = GeM()
        self.bn = nn.BatchNorm1d(self.feat_dim)
        self.num_heads = num_heads

        self.heads = nn.ModuleList()
        for i in range(num_heads):
            # If different_heads is True, only the first head is ArcFace
            if Config.different_heads and i > 0:
                # Simple Linear Head for diversity
                self.heads.append(nn.Linear(self.feat_dim, num_classes))
            else:
                # Standard Balanced ArcFace
                self.heads.append(BalancedArcFace(
                    in_features=self.feat_dim, 
                    out_features=num_classes, 
                    sample_counts=class_counts, 
                    s_base=Config.arcface_s, 
                    m=Config.arcface_m
                ))

    def forward(self, x, label=None):
        features = self.backbone.forward_features(x)
        
        if features.dim() == 3:
            B, N, C = features.shape
            H = W = int(math.sqrt(N))
            if H * W != N: features = features[:, -H*W:, :]
            features = features.permute(0, 2, 1).reshape(B, C, H, W)

        emb = self.gem(features).flatten(1)
        emb = self.bn(emb)
        
        if label is not None:
            logits_list = []
            for head in self.heads:
                # Handle different forward signatures
                if isinstance(head, BalancedArcFace):
                    logits_list.append(head(emb, label))
                else:
                    logits_list.append(head(emb)) # Linear head doesn't take label
            return logits_list
        else:
            return F.normalize(emb, p=2, dim=1)

In [6]:
def collaborative_loss_fn(all_logits, labels, temperature=2.0, beta=0.5):
    """
    beta: Weight for the consensus (agreement) loss.
    temperature: Softens the probability distributions for distillation.
    """
    num_heads = len(all_logits)
    
    # 1. Hard Loss: Individual head accuracy (ArcFace + Label Smoothing)
    # Note: We use the criterion defined in your setup later
    criterion = nn.CrossEntropyLoss(label_smoothing=0.03)
    hard_loss = sum([criterion(logits, labels) for logits in all_logits]) / num_heads
    
    # 2. Consensus Loss: Agreement between heads (KL Divergence)
    soft_loss = 0
    with torch.no_grad():
        # Calculate the ensemble average (teacher distribution)
        avg_probs = torch.stack([F.softmax(l / temperature, dim=1) for l in all_logits]).mean(dim=0)
        
    for logits in all_logits:
        # Each head is forced to match the 'average' opinion of the group
        log_p = F.log_softmax(logits / temperature, dim=1)
        soft_loss += F.kl_div(log_p, avg_probs, reduction='batchmean') * (temperature**2)
    
    soft_loss /= num_heads
    
    return hard_loss + (beta * soft_loss)

In [7]:
# Layer-wise Learning Rate Decay Optimizer
def get_optimizer_params(model, encoder_lr, head_lr, weight_decay=0.0):
    param_optimizer = list(model.named_parameters())
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    optimizer_parameters = []
    
    # Simple LLRD implementation
    layer_decay = 0.9
    num_layers = 24 # Approximate for Large models, or calculate dynamically
    
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        
        lr = encoder_lr
        if "head" in name:
            lr = head_lr
        elif "blocks" in name:
            try:
                # Attempt to extract block index to scale LR
                layer_id = int(name.split("blocks.")[1].split(".")[0])
                lr = encoder_lr * (layer_decay ** (num_layers - layer_id))
            except:
                pass
                
        if any(nd in name for nd in no_decay):
            optimizer_parameters.append({"params": [p], "weight_decay": 0.0, "lr": lr})
        else:
            optimizer_parameters.append({"params": [p], "weight_decay": weight_decay, "lr": lr})
            
    return optimizer_parameters

# Post-Processing Utils (QE & Re-ranking) 
## Added new reranking

In [8]:
def apply_dba(embeddings, k=3):
    # embeddings: (N, D)
    dist = torch.mm(embeddings, embeddings.t())
    _, indices = dist.topk(k, dim=1)
    
    # Average the features of the top-k neighbors
    dba_embeddings = embeddings[indices].mean(dim=1)
    # Re-normalize
    return F.normalize(dba_embeddings, p=2, dim=1)
    
def query_expansion(emb, top_k=3):
    """
    Expands the query by averaging it with its top_k nearest neighbors.
    Uses alpha-weighting (cubed weights) for better rare-class retrieval.
    """
    # Safety Check: Convert PyTorch GPU tensor to NumPy
    if torch.is_tensor(emb):
        emb = emb.detach().cpu().numpy()
        
    print(f"Applying Weighted Query Expansion (k={top_k})...")
    
    # Cosine similarity matrix
    sims = emb @ emb.T
    
    # Get top k indices (descending similarity)
    # np.argsort(-sims) ensures we get the highest similarities first
    indices = np.argsort(-sims, axis=1)[:, :top_k]
    
    new_emb = np.zeros_like(emb)
    for i in range(len(emb)):
        # Similarity scores for the top_k neighbors
        neighbor_sims = sims[i, indices[i]]
        
        # Weighted average logic: Cubing weights emphasizes the closest neighbors
        # This helps 'Bernard' (13 images) by ignoring dissimilar neighbors
        weights = neighbor_sims ** 3  
        
        # Ensure weights sum to 1 to avoid feature scaling issues
        weights_sum = np.sum(weights)
        if weights_sum > 0:
            new_emb[i] = np.sum(emb[indices[i]] * weights[:, np.newaxis], axis=0) / weights_sum
        else:
            new_emb[i] = emb[i] # Fallback if something goes wrong

    # Re-normalize to unit sphere
    norm = np.linalg.norm(new_emb, axis=1, keepdims=True)
    return new_emb / (norm + 1e-12)

def k_reciprocal_rerank(prob, k1=20, k2=6, lambda_value=0.3):
    """
    Re-ranking using k-reciprocal encoding.
    Input 'prob' can be a NumPy array or a PyTorch GPU tensor.
    """
    # Safety Check: Convert to NumPy if it's a Tensor
    if torch.is_tensor(prob):
        prob = prob.detach().cpu().numpy()
        
    print("Applying K-Reciprocal Re-ranking...")
    
    # Distance matrix (Cosine distance = 1 - Cosine similarity)
    q_g_dist = 1 - prob
    original_dist = q_g_dist.copy()
    
    # Sort distances (smallest distance first)
    initial_rank = np.argsort(original_dist, axis=1)
    
    nn_k1 = []
    for i in range(prob.shape[0]):
        # Forward k-nearest neighbors
        forward_k1 = initial_rank[i, :k1 + 1]
        # Backward k-nearest neighbors for the candidates
        backward_k1 = initial_rank[forward_k1, :k1 + 1]
        
        # Find which candidates consider 'i' as a top neighbor (reciprocity)
        fi = np.where(backward_k1 == i)[0]
        nn_k1.append(forward_k1[fi])
        
    jaccard_dist = np.zeros_like(original_dist)
    for i in range(prob.shape[0]):
        # Only look at images within a reasonable distance threshold (0.6)
        # This speeds up calculation and removes extreme outliers
        ind_non_zero = np.where(original_dist[i, :] < 0.6)[0]
        
        # Check mutual neighborhood sets
        ind_images = [inv for inv in ind_non_zero if len(np.intersect1d(nn_k1[i], nn_k1[inv])) > 0]
        
        for j in ind_images:
            intersection = len(np.intersect1d(nn_k1[i], nn_k1[j]))
            union = len(np.union1d(nn_k1[i], nn_k1[j]))
            jaccard_dist[i, j] = 1 - (intersection / (union + 1e-12))
            
    # Combine original distance with Jaccard (re-ranking) distance
    final_dist = jaccard_dist * lambda_value + original_dist * (1 - lambda_value)
    
    # Return as a similarity matrix (1 - distance)
    return 1 - final_dist

def post_process_pipeline(features, names, k_dba=3, k_qe=3, k1=20, k2=6):
    """
    Recommended Order:
    1. Features arrive (Normalized)
    2. Apply DBA (Gallery improves Gallery)
    3. Apply QE (Query expands via improved Gallery)
    4. Apply K-Reciprocal (Final Re-ranking)
    """
    # Convert to torch for fast GPU matrix ops if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    emb = torch.from_numpy(features).to(device)
    
    # 1. Improved DBA (Weighted)
    print("Applying Alpha-Weighted DBA...")
    sims = torch.mm(emb, emb.t())
    # Use alpha=3 weighting for DBA too
    weights, indices = sims.topk(k_dba, dim=1)
    weights = weights.pow(3) 
    
    dba_emb = torch.zeros_like(emb)
    for i in range(len(emb)):
        dba_emb[i] = (weights[i].unsqueeze(1) * emb[indices[i]]).sum(0) / weights[i].sum()
    
    dba_emb = F.normalize(dba_emb, p=2, dim=1)
    
    # 2. Query Expansion (Using DBA-cleaned features)
    # We re-calculate sims using the 'clean' database
    print("Applying Alpha-Weighted QE...")
    sims_qe = torch.mm(dba_emb, dba_emb.t())
    weights_qe, indices_qe = sims_qe.topk(k_qe, dim=1)
    weights_qe = weights_qe.pow(3)
    
    qe_emb = torch.zeros_like(dba_emb)
    for i in range(len(dba_emb)):
        qe_emb[i] = (weights_qe[i].unsqueeze(1) * dba_emb[indices_qe[i]]).sum(0) / weights_qe[i].sum()
        
    qe_emb = F.normalize(qe_emb, p=2, dim=1)
    
    # 3. K-Reciprocal Reranking
    # Pass the similarity matrix (QE_EMB @ QE_EMB.T) to your rerank function
    final_sim_matrix = torch.mm(qe_emb, qe_emb.t()).cpu().numpy()
    
    return k_reciprocal_rerank(final_sim_matrix, k1=k1, k2=k2)

# Training & Inference Engine

In [9]:
@torch.no_grad() # <--- CRITICAL: Prevents gradient memory accumulation
def validate(model, loader):
    model.eval()
    val_loss = 0
    all_feats = []
    all_labels = []
    
    criterion = nn.CrossEntropyLoss()
    
    for imgs, labels in tqdm(loader, desc="Validating", leave=False):
        imgs, labels = imgs.to(Config.device), labels.to(Config.device)
        
        with torch.amp.autocast('cuda'):
            # 1. Get Logits for Loss
            all_logits = model(imgs, labels) 
            loss = sum([criterion(lg, labels) for lg in all_logits]) / len(all_logits)
            val_loss += loss.item()
            
            # 2. Get Features for mAP
            # IMPORTANT: Ensure model returns normalized features here
            # and immediately move to CPU to free GPU memory
            feats = model(imgs) 
            all_feats.append(feats.detach().cpu()) # .detach().cpu() is the safety net
            all_labels.append(labels.detach().cpu())
            
        # Optional: Clear intermediate variables
        del imgs, labels, all_logits, feats

    # Clear GPU cache after the heavy loop
    torch.cuda.empty_cache() 
            
    val_loss /= len(loader)
    
    # --- mAP Calculation (on CPU now) ---
    feats = torch.cat(all_feats, dim=0).numpy()
    labels = torch.cat(all_labels, dim=0).numpy()
    
    # If the similarity matrix is still too big for memory, 
    # we compute it in chunks, but for 371 images, this is fine.
    sim_matrix = feats @ feats.T
    
    aps = []
    for i in range(len(labels)):
        target_label = labels[i]
        true_indices = np.where(labels == target_label)[0]
        true_indices = true_indices[true_indices != i]
        
        if len(true_indices) == 0: continue 
        
        scores = sim_matrix[i]
        order = np.argsort(-scores)
        matches = np.isin(order, true_indices)
        
        pos_ranks = np.where(matches[1:])[0] + 1
        ap = np.sum((np.arange(len(pos_ranks)) + 1) / pos_ranks) / len(true_indices)
        aps.append(ap)
        
    return val_loss, np.mean(aps) if aps else 0.0

def get_current_beta(epoch):
    """Calculates beta based on Config settings"""
    if not Config.use_beta_scheduler:
        return Config.base_beta # Constant beta (e.g., 0.5)
    
    # Linear increase: 0 at epoch 0, reaching Config.base_beta at Config.beta_warmup_epochs
    if epoch < Config.beta_warmup_epochs:
        return (epoch / Config.beta_warmup_epochs) * Config.base_beta
    else:
        return Config.base_beta

# def train_epoch(model, loader, optimizer, scaler, epoch):
#     model.train()
#     loss_meter = 0
    
#     # Calculate beta for this epoch
#     current_beta = get_current_beta(epoch)
    
#     for i, (imgs, labels) in enumerate(tqdm(loader, desc=f"Epoch {epoch} Training", leave=False)):
#         imgs, labels = imgs.to(Config.device), labels.to(Config.device)
        
#         with torch.amp.autocast('cuda'):
#             all_logits = model(imgs, labels)
            
#             # Use the dynamic beta in the loss function
#             loss = collaborative_loss_fn(
#                 all_logits, 
#                 labels, 
#                 temperature=Config.temperature, 
#                 beta=current_beta
#             )
#             loss = loss / Config.grad_accum
            
#         scaler.scale(loss).backward()
        
#         if (i + 1) % Config.grad_accum == 0:
#             scaler.unscale_(optimizer)
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#             scaler.step(optimizer)
#             scaler.update()
#             optimizer.zero_grad()
            
#         loss_meter += loss.item() * Config.grad_accum
        
#     return loss_meter / len(loader), current_beta
    
def train_epoch(model, loader, optimizer, scaler):
    model.train()
    loss_meter = 0
    
    for i, (imgs, labels) in enumerate(tqdm(loader, desc="Training", leave=False)):
        imgs, labels = imgs.to(Config.device), labels.to(Config.device)
        
        with torch.amp.autocast('cuda'):
            # Model returns a list of logits
            all_logits = model(imgs, labels)
            # Use our custom collaborative loss
            loss = collaborative_loss_fn(all_logits, labels, beta=0.5)
            loss = loss / Config.grad_accum
            
        scaler.scale(loss).backward()
        
        if (i + 1) % Config.grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        loss_meter += loss.item() * Config.grad_accum
        
    return loss_meter / len(loader)    
    
@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    feats, names = [], []
    for imgs, fnames in tqdm(loader, desc="Inference"):
        imgs = imgs.to(Config.device)
        
        # Original forward pass
        f1 = model(imgs)
        
        # Test Time Augmentation (Horizontal Flip)
        if Config.use_tta:
            f2 = model(torch.flip(imgs, [3]))
            f1 = (f1 + f2) / 2
            
        feats.append(F.normalize(f1, dim=1).cpu())
        names.extend(fnames)
    return torch.cat(feats, dim=0).numpy(), names

# Execution (Main Loop)

In [10]:
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import WeightedRandomSampler

# --- PATHS ---
TRAIN_CSV = "/kaggle/input/jaguar-re-id/train.csv"
TEST_CSV = "/kaggle/input/jaguar-re-id/test.csv"
TRAIN_DIR = "/kaggle/input/jaguar-re-id/train/train"
TEST_DIR = "/kaggle/input/jaguar-re-id/test/test"

# --- DATA LOADING ---
full_train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# --- SPLIT SETUP (STRATIFIED K-FOLD) ---
skf = StratifiedKFold(n_splits=Config.n_folds, shuffle=True, random_state=Config.seed)
# Create a dummy fold column
full_train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(full_train_df, full_train_df["ground_truth"])):
    full_train_df.loc[val_idx, "fold"] = fold

# Always fit on the FULL set of names
encoder = LabelEncoder()
encoder.fit(full_train_df['ground_truth'])

# Map the global labels
full_train_df['label_idx'] = encoder.transform(full_train_df['ground_truth'])

# --- SELECT DATA AND CALCULATE WEIGHTS ---
if Config.train_full_data:
    print(f"🚀 Training on FULL dataset ({Config.num_classes} classes)")
    train_df = full_train_df.copy()
    
    # Simple global counts
    # We use np.bincount to ensure we get a count for every index from 0 to 30
    counts = np.bincount(train_df['label_idx'], minlength=Config.num_classes)
    
    # Avoid division by zero (though not possible in full data)
    class_weights = 1.0 / np.where(counts == 0, 1, counts)
    sample_weights = class_weights[train_df['label_idx'].values]

else:
    print(f"🔬 Training on FOLD {Config.target_fold} (Validation Mode)")
    # 1. Reset index to prevent the DataLoader IndexError
    train_df = full_train_df[full_train_df["fold"] != Config.target_fold].reset_index(drop=True)
    val_df = full_train_df[full_train_df["fold"] == Config.target_fold].reset_index(drop=True)
    
    # 2. Calculate counts present in THIS fold
    # minlength=Config.num_classes ensures the array is always length 31
    counts = np.bincount(train_df['label_idx'], minlength=Config.num_classes)
    
    # 3. Calculate weights
    # If a class is missing in this fold (count=0), we set weight to 0 
    # so the sampler never tries to pick it.
    class_weights = np.where(counts > 0, 1.0 / counts, 0.0)
    
    # 4. Map weights to the specific rows in our reset train_df
    sample_weights = class_weights[train_df['label_idx'].values]

# --- FINAL SAMPLER ---
sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=len(sample_weights), 
    replacement=True
)

# Convert counts to a list for your CollabEVABoss model
# (Models need to know the frequencies for ArcFace scaling)
class_counts_for_model = counts.tolist()

train_loader = DataLoader(
    JaguarDataset(train_df, TRAIN_DIR, train_transform),
    batch_size=Config.batch_size,
    shuffle=False,
    sampler=sampler, # <--- The Magic Ingredient
    num_workers=2,
    pin_memory=True,
    drop_last=True
)


# --- MODEL SETUP ---
# Initialize with 3 heads (Paper suggested 3-4 for best tradeoff)
#model = EVABoss(num_classes=Config.num_classes, class_counts = class_counts_for_model).to(Config.device)

model = CollabEVABoss(num_classes=Config.num_classes, class_counts=class_counts_for_model, num_heads=3).to(Config.device)

# LLRD optimizer will automatically pick up the 'heads' list
optimizer_params = get_optimizer_params(
    model, 
    encoder_lr=Config.encoder_lr, 
    head_lr=Config.head_lr, 
    weight_decay=Config.weight_decay
)
optimizer = torch.optim.AdamW(optimizer_params)
scaler = torch.amp.GradScaler('cuda')

# The rest of your scheduler and training loop remains the same!
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=Config.num_epochs, eta_min=1e-6
)

#criterion = nn.CrossEntropyLoss(label_smoothing=0.03)

🔬 Training on FOLD 0 (Validation Mode)


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

In [11]:
def save_checkpoint(model, epoch, score, name="EVA-02_best_model.pth"):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'Config': {k: v for k, v in Config.__dict__.items() if not k.startswith("__")},
        'score': score
    }
    torch.save(checkpoint, name)
    print(f"✅ Model saved to {name} with score: {score:.4f}")

In [12]:
# --- MAIN LOOP SETUP ---
if not Config.train_full_data:
    val_dataset = JaguarDataset(val_df, TRAIN_DIR, test_transform)
    val_loader = DataLoader(val_dataset, batch_size=Config.batch_size, shuffle=False, num_workers=2)
else:
    val_loader = None

print(f"🔥 Starting Training: {Config.model_name}")

best_map = 0.0

for epoch in range(Config.num_epochs):
    # 1. Train
    train_loss = train_epoch(model, train_loader, optimizer, scaler)
    scheduler.step()
    
    # 2. Metrics Handling
    current_lr = optimizer.param_groups[0]['lr']
    
    if val_loader is not None:
        # VALIDATION MODE
        val_loss, val_map = validate(model, val_loader)
        print(f"Epoch {epoch+1}/{Config.num_epochs} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"mAP: {val_map:.4f} | LR: {current_lr:.2e}")
        
        # Save best fold model
        if val_map > best_map:
            best_map = val_map
            #save_checkpoint(model, epoch, best_map, is_best=True)
    else:
        # FULL TRAINING MODE
        print(f"Epoch {epoch+1}/{Config.num_epochs} | "
              f"Train Loss: {train_loss:.4f} | LR: {current_lr:.2e}")
        # Just save the latest
        #save_checkpoint(model, epoch, train_loss, is_best=False)

🔥 Starting Training: eva02_large_patch14_448.mim_m38m_ft_in22k_in1k


Training:   0%|          | 0/379 [00:00<?, ?it/s]

Validating:   0%|          | 0/95 [00:00<?, ?it/s]

Epoch 1/13 | Train Loss: 16.9955 | Val Loss: 10.7207 | mAP: 0.5445 | LR: 2.96e-05


Training:   0%|          | 0/379 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/95 [00:00<?, ?it/s]

Epoch 12/13 | Train Loss: 0.7304 | Val Loss: 1.2522 | mAP: 0.9472 | LR: 1.42e-06


Training:   0%|          | 0/379 [00:00<?, ?it/s]

Validating:   0%|          | 0/95 [00:00<?, ?it/s]

Epoch 13/13 | Train Loss: 0.7442 | Val Loss: 1.3268 | mAP: 0.9439 | LR: 1.00e-06


In [ ]:
# --- TRAINING LOOP ---
print(f"🔥 Starting Training: EVA-02 Large | {Config.num_epochs} Epochs")

for epoch in range(Config.num_epochs):
    loss = train_epoch(model, train_loader, optimizer, scaler)
    # loss = train_epoch_sam(model, train_loader, optimizer, criterion, scaler)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{Config.num_epochs} | Loss: {loss:.4f} | LR: {current_lr:.2e}")


# Usage at the end of your training loop:
save_checkpoint(model, epoch=Config.num_epochs, score=loss)

🔥 Starting Training: EVA-02 Large | 13 Epochs


Training:   0%|          | 0/379 [00:00<?, ?it/s]

In [ ]:
def load_checkpoint(filepath, num_classes, class_counts):
    # 1. Load the data from disk
    checkpoint = torch.load(filepath, map_location=Config.device)
    
    # 2. Re-initialize the exact same architecture
    # Note: Use the model class (EVABoss or MegaDBoss) you used for training
    model = CollabEVABoss(
    num_classes=Config.num_classes, 
    class_counts=class_counts_for_model, 
    num_heads=3
).to(Config.device)

    
    # 3. Load the weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(Config.device)
    model.eval() # Set to evaluation mode
    
    print(f"🚀 Loaded model from epoch {checkpoint['epoch']} (Saved score: {checkpoint['score']:.4f})")
    return model

# Usage:
model = load_checkpoint("EVA-02_best_model.pth", Config.num_classes, class_counts_for_model)

In [ ]:

# --- INFERENCE ---
print("\n🔮 Starting Inference...")
unique_test_imgs = sorted(set(test_df["query_image"]) | set(test_df["gallery_image"]))
test_loader = DataLoader(
    JaguarDataset(pd.DataFrame({"filename": unique_test_imgs}), TEST_DIR, test_transform, is_test=True),
    batch_size=Config.batch_size * 2,
    shuffle=False,
    num_workers=2
)

# Extract Features
import torch.nn.functional as F

# 1. Extract Features (Assume emb is torch.Tensor for DBA, then numpy for Rerank)
emb, names = extract_features(model, test_loader) 
img_map = {n: i for i, n in enumerate(names)}
# Note: Ensure emb is on GPU for fast DBA
emb_tensor = torch.from_numpy(emb).to(Config.device)

# --- STEP 1: DATABASE AUGMENTATION (DBA) ---
# We apply DBA to every image in the set first. 
# This "denoises" the features of every jaguar by looking at its neighbors.
if Config.use_dba:
    print("Applying DBA...")
    emb_tensor = apply_dba(emb_tensor, k=3)

# --- STEP 2: QUERY EXPANSION (QE) ---
# Now we enhance the queries using the already-cleaned DBA features.
if Config.use_qe:
    # Using the weighted version helps Bernard (rare classes)
    emb_tensor = query_expansion(emb_tensor, top_k=3)

# Move back to CPU/Numpy for the Re-ranking step
emb_final = emb_tensor#.cpu().numpy()


# --- STEP 3: SIMILARITY & RE-RANKING ---
sim_matrix = emb_final @ emb_final.T

if Config.use_rerank:
    # k-reciprocal usually works better on the "refined" sim_matrix
    sim_matrix = k_reciprocal_rerank(sim_matrix, k1=20, k2=6, lambda_value=0.3)

In [ ]:
# --- GENERATE SUBMISSION ---

preds = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Mapping Predictions"):
    idx_q = img_map[row["query_image"]]
    idx_g = img_map[row["gallery_image"]]
    
    score = sim_matrix[idx_q, idx_g]
    preds.append(max(0.0, min(1.0, score))) # Clip to valid range

sub = pd.DataFrame({"row_id": test_df["row_id"], "similarity": preds})
sub.to_csv("submission.csv", index=False)

print(f"✅ Submission Saved! Mean Similarity: {np.mean(preds):.4f}")